# Teste e Validação Interativa do Pipeline de Selagem Robotizada
Este notebook permite testar o fluxo completo de percepção visual e geração de trajetórias interpoladas diretamente no VS Code.

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Adiciona o diretório raiz ao PYTHONPATH
sys.path.append(os.path.abspath('..'))

from src.preprocessing import preprocess_frame
from src.vision import SealingJointDetector
from src.calibration import transform_points_homography
from src.trajectory import extract_skeleton_points, order_points_tsp, CubicSplineTrajectoryGenerator
from src.utils import plot_pipeline_stages, plot_trajectory_2d

## 1. Criar ou Carregar uma Imagem de Teste

In [ ]:
# Criamos uma junta curva sintética de teste simulando a borda de uma peça aeronáutica
img = np.full((600, 800, 3), 220, dtype=np.uint8)
pts_curve = np.array([
    [100, 150],
    [250, 300],
    [450, 200],
    [600, 400],
    [700, 350]
], np.int32)

# Desenha o cordão/junta com espessura de 12 pixels
cv2.polylines(img, [pts_curve], isClosed=False, color=(40, 40, 40), thickness=12, lineType=cv2.LINE_AA)

plt.figure(figsize=(6, 4))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Imagem de Entrada (Peça / Junta de Selagem)")
plt.axis("off")
plt.show()

## 2. Pré-processamento e Detecção/Segmentação

In [ ]:
# 1. Pré-processamento
prep_img = preprocess_frame(img)

# 2. Detecção com YOLOv8-seg (ou fallback morfológico caso não tenha pesos .pt carregados ainda)
detector = SealingJointDetector()
mask, detections = detector.predict(prep_img)

# 3. Esqueletização da junta (Eixo central com 1 px)
skeleton_pts = extract_skeleton_points(mask)
ordered_pts = order_points_tsp(skeleton_pts)

print(f"Total de pontos no esqueleto extraído: {len(ordered_pts)}")
plot_pipeline_stages(img, prep_img, mask, ordered_pts)

## 3. Conversão para Coordenadas Reais (mm) e Interpolação por B-Spline Cúbica

In [ ]:
# Escala simples de demonstração: 1 pixel = 0.5 mm
scale_px_to_mm = 0.5
points_mm = ordered_pts * scale_px_to_mm

# Subamostragem para pontos de controle suaves
step = max(1, len(points_mm) // 30)
control_points = points_mm[::step]

# Gerador de B-spline cúbica com velocidade constante
spline_gen = CubicSplineTrajectoryGenerator(smoothing_factor=5.0, default_z_mm=10.0)
positions_3d, tangents, normals = spline_gen.fit_spline(control_points, num_samples=150)

# Plota a trajetória com vetores tangentes e normais de orientação
plot_trajectory_2d(control_points, positions_3d, tangents, normals)